In [ ]:
%matplotlib inline
from matplotlib import pyplot as plt

In [ ]:
from recirq.algo_benchmarks.loschmidt import TiltedSquareLatticeLoschmidtSpec, TiltedSquareLatticeLoschmidtData
assert TiltedSquareLatticeLoschmidtData, 'register deserializer'
assert TiltedSquareLatticeLoschmidtSpec, 'register deserializer'


from recirq.cirqflow.quantum_runtime import load_raw_results
run_id = 'testrun-3'
raw_results = load_raw_results(run_id)

In [ ]:
raw_results.executable_results[0].runtime_info

In [ ]:
from recirq.algo_benchmarks.loschmidt import process_results, flat_results_to_dataframe
results = process_results(raw_results)
df = flat_results_to_dataframe(results)

In [ ]:
df

In [ ]:
df.set_index(['run_id', 'width', 'height', 'macrocycle_depth', 'instance_i'])

In [ ]:
import inspect
def group_and_call(func, df):
    func_argnames = list(inspect.signature(func).parameters.keys())
    if 'df' not in func_argnames:
        raise ValueError("group_and_call requires the function `func` to accept a DataFrame argument named `df`")
    func_argnames.remove('df')
    
    def func_wrap(df):
        # the groupby values are in df.name
        kwargs = {argname: df.name[i] for i, argname in enumerate(func_argnames)}
        return func(df=df, **kwargs)
    
    return df.groupby(func_argnames).apply(func_wrap)

In [ ]:
def _plot2(df, width, height, n_qubits):
    gb = df.groupby(['macrocycle_depth']).agg(['mean', 'std']).reset_index()
    plt.errorbar(
        x = gb['macrocycle_depth'],
        y = gb['success_probability', 'mean'],
        yerr = gb['success_probability', 'std'],
        capsize=7,
        marker='o',
        label=f'{n_qubits}q ({width}x{height})'
    )

def _plot1(df, run_id, n_repetitions):
    group_and_call(_plot2, df)
    plt.legend(loc='best', title='width, height')
    plt.yscale('log')
    plt.xlabel('# Cycles per Phi')
    plt.ylabel('Success Probability')
    plt.tight_layout()
    plt.show()
    
    
_ = group_and_call(_plot1, df)

In [ ]:
import numpy as np
def exp_ansatz(macrocycle_depths, a, f):
    return a * np.exp((f - 1.0) * macrocycle_depths)

In [ ]:
def fit():
    """Fit an exponential decay to the collected data."""
    from scipy.optimize import curve_fit

    def fit(cycle, a, f):
        return a * np.exp((f - 1.0) * cycle)

    for i in range(stop):
        (a, f), _ = curve_fit(
            fit,
            xdata=cycle_values,
            ydata=avg_probs[i * step: (i + 1) * step],
        )
        print(f"Error/cycle on qubit configuration {i}: {round((1 - f) * 100, 2)}%")


In [ ]:
from scipy.optimize import curve_fit
import pandas as pd

def _agg_config(df, width, height, n_qubits):    
    (a, f), _ = curve_fit(
        exp_ansatz,
        xdata=df['macrocycle_depth'],
        ydata=df['success_probability'],
    )
    df['a'] = a
    df['f'] = f
    return pd.DataFrame([{
        #'width': width,
        #'height': height,
        #'n_qubits': n_qubits,
        'a': a,
        'f': f,
    }])
def _agg_run(df, run_id, n_repetitions):
    xx = group_and_call(_agg_config, df).droplevel(-1, axis=0)
    return xx
    
    
    
fitted_df = group_and_call(_agg_run, df).reset_index()
fitted_df

In [ ]:
def _plot2(df, width, height, n_qubits):
    gb = df.groupby(['macrocycle_depth']).agg(['mean', 'std']).reset_index()
    g, *_ = plt.errorbar(
        x = gb['macrocycle_depth'],
        y = gb['success_probability', 'mean'],
        yerr = gb['success_probability', 'std'],
        capsize=7,
        marker='o',
        ls='',
        label=f'{n_qubits}q ({width}x{height})'
    )
    
    # ---------------
    #fdf = fitted_df[(fitted_df['run_id'] == run_id)&(fitted_df['n_repetitions'] == n_repetitions)]
    fdf = fitted_df[(fitted_df['width'] == width)&(fitted_df['height'] == height)&(fitted_df['n_qubits'] == n_qubits)]
    assert len(fdf) == 1, fdf
    row = fdf.iloc[0]
    xx = np.linspace(0, 8)
    plt.plot(xx, exp_ansatz(xx, row['a'], row['f']), color=g.get_color(), ls='--', label=f'f = {row["f"]:.2f}')
    

def _plot1(df, run_id, n_repetitions):
    group_and_call(_plot2, df)
    

    
    
    plt.legend(loc='best')
    plt.yscale('log')
    plt.xlabel('# Cycles per Phi')
    plt.ylabel('Success Probability')
    plt.tight_layout()
    plt.show()
    
_ = group_and_call(_plot1, df)